# Ingesting and transforming churn data with Delta Lake and Spark API

<img style="float: right" width="300px" src="https://raw.githubusercontent.com/QuentinAmbard/databricks-demo/main/retail/resources/images/lakehouse-retail/lakehouse-retail-churn-2.png" />

In this notebook, we'll show you an alternative to Spark Declarative Pipelines: building an ingestion pipeline with the Spark API.

As you'll see, this implementation is lower level than the Spark Declarative Pipelines pipeline, and you'll have control over all the implementation details (handling checkpoints, data quality etc).

Lower level also means more power. Using Spark API, you'll have unlimited capabilities to ingest data in Batch or Streaming.

If you're unsure what to use, start with Spark Declarative Pipelines!

*Remember that Databricks workflow can be used to orchestrate a mix of Spark Declarative Pipelines pipeline with standard Spark pipeline.*

As reminder, we have multiple data sources coming from different system:

- Customer profile data *(name, age, adress etc)*
- Orders history *(what our customer bough over time)*
- Events from our application *(when was the last time customers used the application, typically this could be a stream from a Kafka queue)*


Leveraging Spark and Delta Lake makes such an implementation easy.

<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=2162748966026566&notebook=%2F01-Data-ingestion%2Fplain-spark-delta-pipeline%2F01.5-Delta-pipeline-spark-churn&demo_name=lakehouse-retail-c360&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-retail-c360%2F01-Data-ingestion%2Fplain-spark-delta-pipeline%2F01.5-Delta-pipeline-spark-churn&version=1">

In [0]:
%pip install mlflow==2.22.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/29.0 MB ? eta -:--:--
   ━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/29.0 MB 61.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/29.0 MB 260.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━ 20.2/29.0 MB 255.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 29.0/29.0 MB 265.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 29.0/29.0 MB 265.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 29.0/29.0 MB 265.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 29.0/29.0 MB 265.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 29.0/29.0 MB 265.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 29.0/29.0 MB 265.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 29.0/29.0 MB 265.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 29.0/29.0 MB 265.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 29.0/29.

In [0]:
%run ../../_resources/00-setup $reset_all_data=false

USE CATALOG `main__build`
using catalog.database `main__build`.`dbdemos_retail_c360`


data already existing. Run with reset_all_data=true to force a data cleanup for your local demo.


## Building a Spark Data pipeline with Delta Lake

In this example, we'll implement a end 2 end pipeline consuming our customers information. We'll use the medaillon architecture but could build star schema, data vault or any other modelisation.



This can be challenging with traditional systems due to the following:
 * Data quality issue
 * Running concurrent operation
 * Running DELETE/UPDATE/MERGE over files
 * Governance & schema evolution
 * Performance ingesting millions of small files on cloud buckets
 * Processing & analysing unstructured data (image, video...)
 * Switching between batch or streaming depending of your requirement...

## Solving these challenges with Delta Lake

<div style="float:left">

**What's Delta Lake? It's a new OSS standard to bring SQL Transactional database capabilities on top of parquet files!**

Used as a new Spark format, built on top of Spark API / SQL

* **ACID transactions** (Multiple writers can simultaneously modify a data set)
* **Full DML support** (UPDATE/DELETE/MERGE)
* **BATCH and STREAMING** support
* **Data quality** (expectatiosn, Schema Enforcement, Inference and Evolution)
* **TIME TRAVEL** (Look back on how data looked like in the past)
* **Performance boost** with ZOrder, data skipping and Caching, solves small files issue 
</div>


<img src="https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-logo.png" style="height: 200px"/>

<br style="clear: both">

We'll incrementally load new data with the autoloader, enrich this information and then load a model from MLFlow to perform our customer churn prediction.

This information will then be used to build our DBSQL dashboard to track customer behavior and churn.

Let'simplement the following flow: 
 
<div><img width="1100px" src="https://raw.githubusercontent.com/QuentinAmbard/databricks-demo/main/retail/resources/images/lakehouse-retail/lakehouse-retail-churn-de-delta.png"/></div>

*Note that we're including the ML model our [Data Scientist built](TODO) using Databricks AutoML to predict the churn.*

## ![](https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-tiny-logo.png) 1/ Explore the dataset

Let's review the files being received

In [0]:
%sql 
LIST '/Volumes/main/dbdemos_retail_c360/c360/users'

path,name,size,modification_time
/Volumes/main/dbdemos_retail_c360/c360/users/part-00000-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1390-1-c000.json,part-00000-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1390-1-c000.json,232762,1731433134000
/Volumes/main/dbdemos_retail_c360/c360/users/part-00001-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1391-1-c000.json,part-00001-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1391-1-c000.json,232966,1731433134000
/Volumes/main/dbdemos_retail_c360/c360/users/part-00002-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1392-1-c000.json,part-00002-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1392-1-c000.json,232900,1731433134000
/Volumes/main/dbdemos_retail_c360/c360/users/part-00003-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1393-1-c000.json,part-00003-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1393-1-c000.json,232993,1731433134000
/Volumes/main/dbdemos_retail_c360/c360/users/part-00004-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1394-1-c000.json,part-00004-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1394-1-c000.json,234912,1731433134000
/Volumes/main/dbdemos_retail_c360/c360/users/part-00005-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1395-1-c000.json,part-00005-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1395-1-c000.json,234822,1731433134000
/Volumes/main/dbdemos_retail_c360/c360/users/part-00006-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1396-1-c000.json,part-00006-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1396-1-c000.json,235040,1731433134000
/Volumes/main/dbdemos_retail_c360/c360/users/part-00007-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1397-1-c000.json,part-00007-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1397-1-c000.json,235284,1731433134000
/Volumes/main/dbdemos_retail_c360/c360/users/part-00008-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1398-1-c000.json,part-00008-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1398-1-c000.json,234271,1731433134000
/Volumes/main/dbdemos_retail_c360/c360/users/part-00009-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1399-1-c000.json,part-00009-tid-310201503166333536-90ad9cfb-4687-4123-88f0-971517676dd7-1399-1-c000.json,235554,1731433134000


In [0]:
%sql
SELECT * FROM json.`/Volumes/main/dbdemos_retail_c360/c360/users`

address,age_group,canal,churn,country,creation_date,email,firstname,gender,id,last_activity_date,lastname
Unit 0526 Box 0411 DPO AE 20542,7.0,WEBAPP,true,SPAIN,03-06-2014 00:00:00,pittmanlynn@jordan.biz,Kevin,1.0,4822f0b4-76bc-49c6-a419-e559780f7d3c,06-07-2023 05:12:45,Torres
"5375 Dunn Centers Apt. 490 Jasonfort, PR 27149",2.0,WEBAPP,true,USA,08-09-2021 00:00:00,sandovalgerald@oliver.com,Eugene,1.0,4b04e0a8-5bb4-42e6-8bf4-03aed12ad098,06-04-2023 12:19:27,Martin
"4600 Richardson Dale New Wanda, NE 42581",7.0,MOBILE,false,SPAIN,08-12-2021 00:00:00,ybriggs@pierce.org,Andrea,1.0,c941b1c2-501a-43ef-abbf-39fdcae31c65,06-06-2023 20:54:21,Pierce
"99916 Fleming Ford North Aaron, MN 04354",9.0,WEBAPP,false,SPAIN,08-04-2021 00:00:00,ericksonbeth@pacheco.com,Joel,1.0,d99d1fa7-b55f-4e2a-87e7-b664edd7b1ce,06-04-2023 03:07:32,Young
"81869 Jones Spur Chavezchester, PW 96179",0.0,WEBAPP,true,FR,07-22-2021 00:00:00,morenodaniel@bates-caldwell.biz,Christopher,0.0,2cc1c034-4862-4046-a0a1-aa53eb705339,06-09-2023 19:59:13,Miller
"42383 Erickson Ford Suite 763 Mackport, MP 77823",3.0,PHONE,true,FR,08-17-2021 00:00:00,nathaniel61@rogers-myers.com,Kathleen,1.0,d37df85d-fb2e-4266-9f0f-1a87fc024a62,06-07-2023 11:58:21,Hodges
"4149 Farmer Trace New Lauren, FM 85356",8.0,WEBAPP,true,USA,08-16-2021 00:00:00,martineznatasha@parsons-roberts.com,Scott,0.0,6bb145a3-4b3b-4407-b7b9-733dd02a16b8,06-08-2023 20:56:00,Alexander
"2998 Scott Overpass East Williamville, GA 19474",10.0,WEBAPP,true,USA,08-07-2021 00:00:00,jenniferthompson@johnson.info,Todd,1.0,f785c533-affb-4d7a-a0be-f1dec5596816,06-05-2023 02:12:09,Tucker
"020 Lisa Summit Ellisonbury, RI 29406",1.0,PHONE,true,SPAIN,07-26-2021 00:00:00,oliviaavery@ramirez.com,Rebecca,1.0,fec11eb7-276d-4419-84c0-aa5e1bac3768,06-03-2023 13:38:27,Mckee
"0211 Williams Lane Castroland, WY 86447",8.0,WEBAPP,false,FR,08-07-2021 00:00:00,jacksonadam@rice-hardin.org,Sergio,1.0,708c7f78-a133-4074-b370-37e3c2edc706,06-03-2023 13:47:01,Griffin


### 1/ Loading our data using Databricks Autoloader (cloud_files)
<div style="float:right">
  <img width="700px" src="https://raw.githubusercontent.com/QuentinAmbard/databricks-demo/main/retail/resources/images/lakehouse-retail/lakehouse-retail-churn-de-delta-1.png"/>
</div>
  
Autoloader allow us to efficiently ingest millions of files from a cloud storage, and support efficient schema inference and evolution at scale.

For more details on autoloader, run `dbdemos.install('auto-loader')`

Let's use it to [create our pipeline](https://e2-demo-field-eng.cloud.databricks.com/?o=1444828305810485#joblist/pipelines/95f28631-1884-425e-af69-05c3f397dd90) and ingest the raw JSON & CSV data being delivered in our blob storage `/demos/retail/churn/...`. 

In [0]:
%sql
-- Note: tables are automatically created during  .writeStream.table("user_bronze") operation, but we can also use plain SQL to create them:
CREATE TABLE IF NOT EXISTS spark_churn_users_bronze (
     id                 STRING,
     email              STRING,
     creation_date      STRING,
     last_activity_date STRING,
     firstname          STRING,
     lastname           STRING,
     address            STRING,
     age_group          DOUBLE,
     canal              STRING,
     churn              BOOLEAN,
     country            STRING,
     gender             DOUBLE,
     _rescued_data      STRING
  ) 
  USING DELTA 
  CLUSTER BY (firstname, lastname) -- accelerate query by firstname/lastname with Liquid
  TBLPROPERTIES (
     delta.autooptimize.optimizewrite = TRUE,
     delta.autooptimize.autocompact   = TRUE ); 
-- With these 2 last options, Databricks engine will solve small files & optimize write out of the box!

In [0]:
volume_folder = f"/Volumes/{catalog}/{db}/c360"

def ingest_folder(folder, data_format, table):
  bronze_products = (spark.readStream
                              .format("cloudFiles")
                              .option("cloudFiles.format", data_format)
                              .option("cloudFiles.inferColumnTypes", "true")
                              .option("cloudFiles.schemaLocation", f"{volume_folder}/schema_spark/{table}") #Autoloader will automatically infer all the schema & evolution
                              .load(folder))

  return (bronze_products.writeStream
                    .option("checkpointLocation", f"{volume_folder}/checkpoint_spark/{table}") #exactly once delivery on Delta tables over restart/kill
                    .option("mergeSchema", "true") #merge any new column dynamically
                    .trigger(availableNow = True) #Remove for real time streaming
                    .table(table)) #Table will be created if we haven't specified the schema first
  
ingest_folder(f'{volume_folder}/orders', 'json', 'spark_churn_orders_bronze')
ingest_folder(f'{volume_folder}/events', 'csv', 'spark_churn_app_events')
ingest_folder(f'{volume_folder}/users', 'json',  'spark_churn_users_bronze').awaitTermination()

In [0]:
%sql 
-- Note the "_rescued_data" column. If we receive wrong data not matching existing schema, it'll be stored here
select * from spark_churn_users_bronze;

id,email,creation_date,last_activity_date,firstname,lastname,address,age_group,canal,churn,country,gender,_rescued_data
a5dd3f01-71c6-424b-bf23-8fbdd9704db9,sbrooks@hebert.com,04-19-2013 00:00:00,06-04-2023 16:55:26,Christopher,Davis,"0866 Luna Crest Briggsside, AR 02197",10.0,WEBAPP,true,USA,1.0,null
782a5365-708b-4963-b834-01ad710ec86f,opatterson@moore.biz,08-07-2021 00:00:00,06-05-2023 18:15:01,James,Gonzalez,"58770 Cathy Station Suite 272 Huertaland, WV 24449",4.0,WEBAPP,true,FR,1.0,null
ce88c719-6abb-4a10-8dcb-49f34ad0bd81,matthewwhite@fox.com,08-03-2021 00:00:00,06-02-2023 22:36:41,Aaron,Smith,"PSC 5633, Box 7217 APO AP 72279",4.0,WEBAPP,true,USA,1.0,null
0f87eb48-6015-4758-8ca9-808d9dc5d8b4,dsmith@oconnor-newton.com,07-25-2021 00:00:00,06-04-2023 12:16:24,Brian,Williams,"6857 Potts Square Harrymouth, AL 52721",3.0,WEBAPP,true,USA,1.0,null
c289d5de-f070-4019-a131-23e3b4473a9e,charles53@garcia.com,08-13-2021 00:00:00,06-04-2023 15:17:23,Gary,Jackson,"8351 Turner Inlet Suite 215 New Anthonyside, IN 71953",8.0,WEBAPP,true,USA,1.0,null
e122e99f-b09b-4555-b44e-d26c732200f3,emmaherrera@wells.biz,08-15-2021 00:00:00,06-04-2023 22:37:22,Matthew,Rogers,"9203 Jason Forge Maryview, MS 79947",2.0,WEBAPP,true,USA,1.0,null
28576200-0530-4370-ac21-5e1be93b9b6b,rriggs@armstrong.org,08-05-2021 00:00:00,06-08-2023 19:56:16,William,Romero,"3501 Wright Ridge Suite 542 Johnsonside, MP 03305",6.0,WEBAPP,true,FR,1.0,null
43034f6b-8e2f-4895-877e-cffb44dfc26b,rriggs@armstrong.org,08-05-2021 00:00:00,06-08-2023 19:56:16,William,Romero,"3501 Wright Ridge Suite 542 Johnsonside, MP 03305",1.0,WEBAPP,false,FR,0.0,null
25bf31a3-02e9-46e4-a591-6c1c476cc686,matthew96@miller.com,08-03-2021 00:00:00,06-04-2023 13:42:34,Marissa,Wall,"68012 Tiffany Forges Mitchellberg, MN 48959",7.0,PHONE,true,USA,1.0,null
10f23980-4c69-437d-9816-e42bf3792acd,douglasschultz@gilmore-matthews.com,08-02-2021 00:00:00,06-01-2023 03:34:03,Taylor,Harrison,Unit 0690 Box 4449 DPO AA 14319,3.0,WEBAPP,true,FR,1.0,null



## ![](https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-tiny-logo.png) 2/ Silver data: anonimized table, date cleaned

<img width="700px" style="float:right" src="https://raw.githubusercontent.com/QuentinAmbard/databricks-demo/main/retail/resources/images/lakehouse-retail/lakehouse-retail-churn-de-delta-2.png"/>

We can chain these incremental transformation between tables, consuming only new data.

This can be triggered in near realtime, or in batch fashion, for example as a job running every night to consume daily data.

In [0]:
(spark.readStream 
        .table("spark_churn_users_bronze")
        .withColumnRenamed("id", "user_id")
        .withColumn("email", sha1(col("email")))
        .withColumn("creation_date", to_timestamp(col("creation_date"), "MM-dd-yyyy H:mm:ss"))
        .withColumn("last_activity_date", to_timestamp(col("last_activity_date"), "MM-dd-yyyy HH:mm:ss"))
        .withColumn("firstname", initcap(col("firstname")))
        .withColumn("lastname", initcap(col("lastname")))
        .withColumn("age_group", col("age_group").cast('int'))
        .withColumn("gender", col("gender").cast('int'))
        .withColumn("churn", col("churn").cast('int'))
        .drop(col("_rescued_data"))
     .writeStream
        .option("checkpointLocation", f"{volume_folder}/checkpoint_spark/churn_users")
        .option("mergeSchema", "true")
        .trigger(availableNow = True)
        .table("spark_churn_users").awaitTermination())

In [0]:
%sql select * from spark_churn_users;

user_id,email,creation_date,last_activity_date,firstname,lastname,address,age_group,canal,churn,country,gender
782a5365-708b-4963-b834-01ad710ec86f,93b1bfb58e4e9d9475be2e303fb76c9477c354bc,2021-08-07T00:00:00Z,2023-06-05T18:15:01Z,James,Gonzalez,"58770 Cathy Station Suite 272 Huertaland, WV 24449",4,WEBAPP,1,FR,1
ce88c719-6abb-4a10-8dcb-49f34ad0bd81,6064dcce588ac8f1abc7e40290c02bc74e7e4b0f,2021-08-03T00:00:00Z,2023-06-02T22:36:41Z,Aaron,Smith,"PSC 5633, Box 7217 APO AP 72279",4,WEBAPP,1,USA,1
0f87eb48-6015-4758-8ca9-808d9dc5d8b4,74ad3ac6d6e8fb4e0d2943443934e1f250d470f3,2021-07-25T00:00:00Z,2023-06-04T12:16:24Z,Brian,Williams,"6857 Potts Square Harrymouth, AL 52721",3,WEBAPP,1,USA,1
c289d5de-f070-4019-a131-23e3b4473a9e,dc93b10c1c0faf2577258491c66e57ece19684a9,2021-08-13T00:00:00Z,2023-06-04T15:17:23Z,Gary,Jackson,"8351 Turner Inlet Suite 215 New Anthonyside, IN 71953",8,WEBAPP,1,USA,1
e122e99f-b09b-4555-b44e-d26c732200f3,6aab4a9aca1b6133799a46c050c0ed840a8b696e,2021-08-15T00:00:00Z,2023-06-04T22:37:22Z,Matthew,Rogers,"9203 Jason Forge Maryview, MS 79947",2,WEBAPP,1,USA,1
28576200-0530-4370-ac21-5e1be93b9b6b,88ee42714f43e273f0deb5c29bd987e4a80f452c,2021-08-05T00:00:00Z,2023-06-08T19:56:16Z,William,Romero,"3501 Wright Ridge Suite 542 Johnsonside, MP 03305",6,WEBAPP,1,FR,1
43034f6b-8e2f-4895-877e-cffb44dfc26b,88ee42714f43e273f0deb5c29bd987e4a80f452c,2021-08-05T00:00:00Z,2023-06-08T19:56:16Z,William,Romero,"3501 Wright Ridge Suite 542 Johnsonside, MP 03305",1,WEBAPP,0,FR,0
25bf31a3-02e9-46e4-a591-6c1c476cc686,0b7a9e799fcb6b7809dec8ea158fa46627258c7d,2021-08-03T00:00:00Z,2023-06-04T13:42:34Z,Marissa,Wall,"68012 Tiffany Forges Mitchellberg, MN 48959",7,PHONE,1,USA,1
10f23980-4c69-437d-9816-e42bf3792acd,afb706c48c703a876568b2c1e1a6edd1a4d32fe8,2021-08-02T00:00:00Z,2023-06-01T03:34:03Z,Taylor,Harrison,Unit 0690 Box 4449 DPO AA 14319,3,WEBAPP,1,FR,1
0b7ec6ea-a8d6-4db3-85ee-c7ef1de89d21,b67cd6635b8f0faf23b6e0c0954bd4877c1d1bd8,2021-07-28T00:00:00Z,2023-06-08T15:41:30Z,Richard,Sullivan,"761 Anderson Squares East Roseton, MN 33616",5,WEBAPP,1,FR,1


In [0]:
(spark.readStream 
        .table("spark_churn_orders_bronze")
        .withColumnRenamed("id", "order_id")
        .withColumn("amount", col("amount").cast('int'))
        .withColumn("item_count", col("item_count").cast('int'))
        .withColumn("creation_date", to_timestamp(col("transaction_date"), "MM-dd-yyyy H:mm:ss"))
        .drop(col("_rescued_data"))
     .writeStream
        .option("checkpointLocation", f"{volume_folder}/checkpoint_spark/churn_orders")
        .option("mergeSchema", "true")
        .trigger(availableNow = True)
        .table("spark_churn_orders").awaitTermination())

### 3/ Aggregate and join data to create our ML features

<img width="700px" style="float:right" src="https://raw.githubusercontent.com/QuentinAmbard/databricks-demo/main/retail/resources/images/lakehouse-retail/lakehouse-retail-churn-de-delta-3.png"/>


We're now ready to create the features required for our Churn prediction.

We need to enrich our user dataset with extra information which our model will use to help predicting churn, sucj as:

* last command date
* number of item bought
* number of actions in our website
* device used (ios/iphone)
* ...

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE spark_churn_features AS
      WITH 
          spark_churn_orders_stats AS (SELECT user_id, count(*) as order_count, sum(amount) as total_amount, sum(item_count) as total_item, max(creation_date) as last_transaction
            FROM spark_churn_orders GROUP BY user_id),  
          spark_churn_app_events_stats as (
            SELECT first(platform) as platform, user_id, count(*) as event_count, count(distinct session_id) as session_count, max(to_timestamp(date, "MM-dd-yyyy HH:mm:ss")) as last_event
              FROM spark_churn_app_events GROUP BY user_id)
        SELECT *, 
           datediff(now(), creation_date) as days_since_creation,
           datediff(now(), last_activity_date) as days_since_last_activity,
           datediff(now(), last_event) as days_last_event
           FROM spark_churn_users
             INNER JOIN spark_churn_orders_stats using (user_id)
             INNER JOIN spark_churn_app_events_stats using (user_id)""")
     
display(spark.table("spark_churn_features"))

user_id,email,creation_date,last_activity_date,firstname,lastname,address,age_group,canal,churn,country,gender,order_count,total_amount,total_item,last_transaction,platform,event_count,session_count,last_event,days_since_creation,days_since_last_activity,days_last_event
abbdf2ee-cedd-4176-853e-3e0b99b450c7,33cc7abf40895d004ff14849ed3a446adcfd87ad,2022-01-13T00:00:00Z,2023-06-07T22:45:56Z,Kyle,Parker,"22334 Stacey Valley Suite 296 Jasonland, WI 99608",10,WEBAPP,1,FR,1,6,344,11,2023-06-09T03:01:10Z,android,6,6,2023-06-02T17:35:37Z,1426,916,921
66db6742-5460-4f0e-a503-1652d9d40296,95ea0551c48706b6bdc2004478bbd0a33492c642,2022-03-16T00:00:00Z,2023-06-08T09:01:25Z,Lee,Edwards,"846 Smith Ridges Apt. 149 South Elizabeth, MD 93813",6,WEBAPP,0,USA,1,4,310,11,2023-06-06T14:43:31Z,other,4,3,2023-06-02T10:55:34Z,1364,915,921
194bfcef-5a12-448e-af92-576b76392933,ea2fa260e87ca3d080afebdd53984429e993f60a,2021-12-25T00:00:00Z,2023-06-06T19:14:42Z,Rhonda,Gaines,"05732 Robert Fields Apt. 789 Port Kaylafort, AR 35664",8,PHONE,1,SPAIN,0,5,253,10,2023-06-07T13:31:29Z,ios,5,4,2023-06-03T11:45:48Z,1445,917,920
df13bf73-1976-4495-b0cf-a2dfac5231d6,4c9d435c969c93c532eb0a752c0a05b568520b45,2021-09-06T00:00:00Z,2023-06-05T02:12:09Z,Todd,Tucker,"2998 Scott Overpass East Williamville, GA 19474",0,WEBAPP,1,SPAIN,1,5,235,9,2023-06-07T07:59:35Z,other,5,5,2023-06-08T22:33:12Z,1555,918,915
e854931d-d210-4693-8190-3d2e4c580596,98ab65890d5087d383ca4fa91f22a2d5a24a4c31,2021-07-29T00:00:00Z,2023-06-06T16:34:34Z,Ernest,Moore,"127 Webb Drive Elizabethbury, CT 65649",2,WEBAPP,1,FR,1,3,132,5,2023-06-06T14:42:19Z,ios,3,3,2023-06-09T07:44:39Z,1594,917,914
64817960-6409-4149-bb0b-bd9f81020a5b,fb87b78b9f0a7fe141cbc305f97e7b0f1e5f651d,2021-12-31T00:00:00Z,2023-06-02T08:41:10Z,Patricia,Mcclain,"9889 Decker Village West Maryfort, AS 89347",2,PHONE,1,USA,1,3,255,8,2023-06-06T14:33:52Z,ios,3,3,2023-06-04T14:13:05Z,1439,921,919
6a7e3819-c475-42aa-906d-5dfdfd3f6706,95ea0551c48706b6bdc2004478bbd0a33492c642,2022-02-14T00:00:00Z,2023-06-08T09:01:25Z,Lee,Edwards,"846 Smith Ridges Apt. 149 South Elizabeth, MD 93813",8,WEBAPP,0,FR,1,3,116,6,2023-06-08T11:27:20Z,other,3,2,2023-06-02T14:34:54Z,1394,915,921
e8c05952-8898-42af-9f25-5931f11520ca,df52b442625df2e109d1ca2a894efe6dfe82115e,2022-03-02T00:00:00Z,2023-06-06T00:50:01Z,David,Anderson,"66475 Norma Island Apt. 113 Jessicahaven, MH 19301",5,WEBAPP,1,USA,1,3,205,7,2023-06-09T03:35:48Z,ios,3,3,2023-06-08T16:47:39Z,1378,917,915
e1162688-b147-49b1-a514-ca258df869b4,997c024f7a5c5a5dd2a509dc584b7aadba0e29f3,2022-04-11T00:00:00Z,2023-06-08T16:27:48Z,David,Martinez,"533 Quinn Shoal Moorechester, SD 41317",8,WEBAPP,0,USA,1,3,188,6,2023-06-09T00:51:46Z,other,3,2,2023-06-07T16:38:39Z,1338,915,916
5f358cf8-2867-4ec8-aae1-1b371f9e6062,9c03e6022dd3a2005be3f6e43187620e4811b881,2021-12-12T00:00:00Z,2023-06-01T13:47:23Z,Linda,Lara,"978 Dodson Camp Cookmouth, IN 17720",4,PHONE,1,SPAIN,0,5,330,13,2023-06-09T14:36:35Z,ios,5,5,2023-06-09T06:57:40Z,1458,922,914


## 5/ Enriching the gold data with a ML model

<img width="700px" style="float:right" src="https://raw.githubusercontent.com/QuentinAmbard/databricks-demo/main/retail/resources/images/lakehouse-retail/lakehouse-retail-churn-de-delta-5.png"/>

Our Data scientist team has build a churn prediction model using Auto ML and saved it into Databricks Model registry. 

One of the key value of the Lakehouse is that we can easily load this model and predict our churn right into our pipeline. 

Note that we don't have to worry about the model framework (sklearn or other), MLFlow abstract that for us.

In [0]:
import mlflow
# Setup registry to use Databricks Unity Catalog
mlflow.set_registry_uri('databricks-uc')

#                                                                                            Alias/version
#                                                                 Model name (UC)                   |   
#                                                                     |                             |   
predict_churn_udf = mlflow.pyfunc.spark_udf(spark, f"models:/{catalog}.{db}.dbdemos_customer_churn@prod", result_type="long", env_manager='virtualenv')

2025/12/09 16:37:49 INFO mlflow.pyfunc: This UDF will use virtualenv to recreate the model's software environment for inference. This may take extra time during execution.


2025/12/09 16:37:49 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
2025/12/09 16:37:49 INFO mlflow.utils.virtualenv: Installing python 3.12.3 if it does not exist
2025/12/09 16:39:03 INFO mlflow.utils.virtualenv: Creating a new environment in /local_disk0/.ephemeral_nfs/repl_tmp_data/ReplId-19b03-f8419-9/mlflow/envs/virtualenv_envs/mlflow-2e7e6045218a336288d5eded6dc6233a3bccf3c4 with /local_disk0/.ephemeral_nfs/repl_tmp_data/ReplId-19b03-f8419-9/mlflow/envs/pyenv_root/versions/3.12.3/bin/python
2025/12/09 16:39:03 INFO mlflow.utils.virtualenv: Installing dependencies
2025/12/09 16:39:56 INFO mlflow.utils.environment: === Running command '['bash', '-c', 'source /local_disk0/.ephemeral_nfs/repl_tmp_data/ReplId-19b03-f8419-9/mlflow/envs/virtualenv_envs/mlflow-2e7e6045218a336288d5eded6dc6233a3bccf3c4/bin/activate && python -c ""']'


In [0]:
columns = predict_churn_udf.metadata.get_input_schema().input_names()
predictions = spark.table('spark_churn_features').limit(10).withColumn('churn_prediction', predict_churn_udf(*columns))
display(predictions)

user_id,email,creation_date,last_activity_date,firstname,lastname,address,age_group,canal,churn,country,gender,order_count,total_amount,total_item,last_transaction,platform,event_count,session_count,last_event,days_since_creation,days_since_last_activity,days_last_event,churn_prediction
abbdf2ee-cedd-4176-853e-3e0b99b450c7,33cc7abf40895d004ff14849ed3a446adcfd87ad,2022-01-13T00:00:00Z,2023-06-07T22:45:56Z,Kyle,Parker,"22334 Stacey Valley Suite 296 Jasonland, WI 99608",10,WEBAPP,1,FR,1,6,344,11,2023-06-09T03:01:10Z,android,6,6,2023-06-02T17:35:37Z,1426,916,921,1
66db6742-5460-4f0e-a503-1652d9d40296,95ea0551c48706b6bdc2004478bbd0a33492c642,2022-03-16T00:00:00Z,2023-06-08T09:01:25Z,Lee,Edwards,"846 Smith Ridges Apt. 149 South Elizabeth, MD 93813",6,WEBAPP,0,USA,1,4,310,11,2023-06-06T14:43:31Z,other,4,3,2023-06-02T10:55:34Z,1364,915,921,1
194bfcef-5a12-448e-af92-576b76392933,ea2fa260e87ca3d080afebdd53984429e993f60a,2021-12-25T00:00:00Z,2023-06-06T19:14:42Z,Rhonda,Gaines,"05732 Robert Fields Apt. 789 Port Kaylafort, AR 35664",8,PHONE,1,SPAIN,0,5,253,10,2023-06-07T13:31:29Z,ios,5,4,2023-06-03T11:45:48Z,1445,917,920,1
df13bf73-1976-4495-b0cf-a2dfac5231d6,4c9d435c969c93c532eb0a752c0a05b568520b45,2021-09-06T00:00:00Z,2023-06-05T02:12:09Z,Todd,Tucker,"2998 Scott Overpass East Williamville, GA 19474",0,WEBAPP,1,SPAIN,1,5,235,9,2023-06-07T07:59:35Z,other,5,5,2023-06-08T22:33:12Z,1555,918,915,1
e854931d-d210-4693-8190-3d2e4c580596,98ab65890d5087d383ca4fa91f22a2d5a24a4c31,2021-07-29T00:00:00Z,2023-06-06T16:34:34Z,Ernest,Moore,"127 Webb Drive Elizabethbury, CT 65649",2,WEBAPP,1,FR,1,3,132,5,2023-06-06T14:42:19Z,ios,3,3,2023-06-09T07:44:39Z,1594,917,914,1
64817960-6409-4149-bb0b-bd9f81020a5b,fb87b78b9f0a7fe141cbc305f97e7b0f1e5f651d,2021-12-31T00:00:00Z,2023-06-02T08:41:10Z,Patricia,Mcclain,"9889 Decker Village West Maryfort, AS 89347",2,PHONE,1,USA,1,3,255,8,2023-06-06T14:33:52Z,ios,3,3,2023-06-04T14:13:05Z,1439,921,919,1
6a7e3819-c475-42aa-906d-5dfdfd3f6706,95ea0551c48706b6bdc2004478bbd0a33492c642,2022-02-14T00:00:00Z,2023-06-08T09:01:25Z,Lee,Edwards,"846 Smith Ridges Apt. 149 South Elizabeth, MD 93813",8,WEBAPP,0,FR,1,3,116,6,2023-06-08T11:27:20Z,other,3,2,2023-06-02T14:34:54Z,1394,915,921,0
e8c05952-8898-42af-9f25-5931f11520ca,df52b442625df2e109d1ca2a894efe6dfe82115e,2022-03-02T00:00:00Z,2023-06-06T00:50:01Z,David,Anderson,"66475 Norma Island Apt. 113 Jessicahaven, MH 19301",5,WEBAPP,1,USA,1,3,205,7,2023-06-09T03:35:48Z,ios,3,3,2023-06-08T16:47:39Z,1378,917,915,1
e1162688-b147-49b1-a514-ca258df869b4,997c024f7a5c5a5dd2a509dc584b7aadba0e29f3,2022-04-11T00:00:00Z,2023-06-08T16:27:48Z,David,Martinez,"533 Quinn Shoal Moorechester, SD 41317",8,WEBAPP,0,USA,1,3,188,6,2023-06-09T00:51:46Z,other,3,2,2023-06-07T16:38:39Z,1338,915,916,1
5f358cf8-2867-4ec8-aae1-1b371f9e6062,9c03e6022dd3a2005be3f6e43187620e4811b881,2021-12-12T00:00:00Z,2023-06-01T13:47:23Z,Linda,Lara,"978 Dodson Camp Cookmouth, IN 17720",4,PHONE,1,SPAIN,0,5,330,13,2023-06-09T14:36:35Z,ios,5,5,2023-06-09T06:57:40Z,1458,922,914,1


## Simplify your operations with transactional DELETE/UPDATE/MERGE operations

Traditional Data Lake struggle to run these simple DML operations. Using Databricks and Delta Lake, your data is stored on your blob storage with transactional capabilities. You can issue DML operation on Petabyte of data without having to worry about concurrent operations.

In [0]:
%sql DELETE FROM spark_churn_users where creation_date < '2016-01-01T03:38:55.000+0000';

num_affected_rows
0


In [0]:
%sql describe history spark_churn_users;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
130,2025-12-09T16:40:07Z,7644138420879474,quentin.ambard@databricks.com,DELETE,"Map(predicate -> [""(creation_date#3724 < 2016-01-01 03:38:55)""])","List(952177213393021, field-demos_lakehouse-retail-c360, 414223767184716, 208424332425007, 7644138420879474, manual)",null,1209-163045-y0yryoku,129,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 531, numDeletionVectorsUpdated -> 0, numDeletedRows -> 0, scanTimeMs -> 516, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/16.4.x-cpu-ml-scala2.12
129,2025-12-05T21:40:54Z,7644138420879474,quentin.ambard@databricks.com,SET TBLPROPERTIES,"Map(properties -> {""delta.autoOptimize.optimizeWrite"":""true"",""delta.autoOptimize.autoCompact"":""true""})","List(952177213393021, field-demos_lakehouse-retail-c360, 875683099325679, 27033078850470, 7644138420879474, manual)",null,1205-213631-x9t236z6,128,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-cpu-ml-scala2.12
128,2025-12-05T21:40:49Z,7644138420879474,quentin.ambard@databricks.com,DELETE,"Map(predicate -> [""(creation_date#3724 < 2016-01-01 03:38:55)""])","List(952177213393021, field-demos_lakehouse-retail-c360, 875683099325679, 27033078850470, 7644138420879474, manual)",null,1205-213631-x9t236z6,127,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 507, numDeletionVectorsUpdated -> 0, numDeletedRows -> 0, scanTimeMs -> 494, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/16.4.x-cpu-ml-scala2.12
127,2025-12-05T21:28:20Z,7644138420879474,quentin.ambard@databricks.com,SET TBLPROPERTIES,"Map(properties -> {""delta.autoOptimize.optimizeWrite"":""true"",""delta.autoOptimize.autoCompact"":""true""})","List(952177213393021, field-demos_lakehouse-retail-c360, 942113127256950, 350213648955481, 7644138420879474, manual)",null,1205-211904-mg9fxfli,126,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-cpu-ml-scala2.12
126,2025-12-05T21:28:15Z,7644138420879474,quentin.ambard@databricks.com,DELETE,"Map(predicate -> [""(creation_date#3724 < 2016-01-01 03:38:55)""])","List(952177213393021, field-demos_lakehouse-retail-c360, 942113127256950, 350213648955481, 7644138420879474, manual)",null,1205-211904-mg9fxfli,125,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 502, numDeletionVectorsUpdated -> 0, numDeletedRows -> 0, scanTimeMs -> 489, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/16.4.x-cpu-ml-scala2.12
125,2025-12-05T19:38:44Z,7644138420879474,quentin.ambard@databricks.com,SET TBLPROPERTIES,"Map(properties -> {""delta.autoOptimize.optimizeWrite"":""true"",""delta.autoOptimize.autoCompact"":""true""})","List(952177213393021, field-demos_lakehouse-retail-c360, 1064340896756889, 294420372728475, 7644138420879474, manual)",null,1205-192935-78gur8ya,124,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-cpu-ml-scala2.12
124,2025-12-05T19:38:39Z,7644138420879474,quentin.ambard@databricks.com,DELETE,"Map(predicate -> [""(creation_date#3724 < 2016-01-01 03:38:55)""])","List(952177213393021, field-demos_lakehouse-retail-c360, 1064340896756889, 294420372728475, 7644138420879474, manual)",null,1205-192935-78gur8ya,123,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -

In [0]:
%sql 
 --also works with AS OF TIMESTAMP "yyyy-MM-dd HH:mm:ss"
select * from spark_churn_users version as of 1 ;

-- You made the DELETE by mistake ? You can easily restore the table at a given version / date:
-- RESTORE TABLE spark_churn_users_clone TO VERSION AS OF 1

-- Or clone it (SHALLOW provides zero copy clone):
-- CREATE TABLE spark_user_gold_clone SHALLOW|DEEP CLONE user_gold VERSION AS OF 1

-- Turn on CDC to capture insert/update/delete operation:
-- ALTER TABLE myDeltaTable SET TBLPROPERTIES (delta.enableChangeDataFeed = true)

user_id,email,creation_date,last_activity_date,firstname,lastname,address,age_group,canal,churn,country,gender
a5dd3f01-71c6-424b-bf23-8fbdd9704db9,ca617a8e820815654c96802e691d5b259ffd7c11,2013-04-19T00:00:00Z,2023-06-04T16:55:26Z,Christopher,Davis,"0866 Luna Crest Briggsside, AR 02197",10,WEBAPP,1,USA,1
782a5365-708b-4963-b834-01ad710ec86f,93b1bfb58e4e9d9475be2e303fb76c9477c354bc,2021-08-07T00:00:00Z,2023-06-05T18:15:01Z,James,Gonzalez,"58770 Cathy Station Suite 272 Huertaland, WV 24449",4,WEBAPP,1,FR,1
ce88c719-6abb-4a10-8dcb-49f34ad0bd81,6064dcce588ac8f1abc7e40290c02bc74e7e4b0f,2021-08-03T00:00:00Z,2023-06-02T22:36:41Z,Aaron,Smith,"PSC 5633, Box 7217 APO AP 72279",4,WEBAPP,1,USA,1
0f87eb48-6015-4758-8ca9-808d9dc5d8b4,74ad3ac6d6e8fb4e0d2943443934e1f250d470f3,2021-07-25T00:00:00Z,2023-06-04T12:16:24Z,Brian,Williams,"6857 Potts Square Harrymouth, AL 52721",3,WEBAPP,1,USA,1
c289d5de-f070-4019-a131-23e3b4473a9e,dc93b10c1c0faf2577258491c66e57ece19684a9,2021-08-13T00:00:00Z,2023-06-04T15:17:23Z,Gary,Jackson,"8351 Turner Inlet Suite 215 New Anthonyside, IN 71953",8,WEBAPP,1,USA,1
e122e99f-b09b-4555-b44e-d26c732200f3,6aab4a9aca1b6133799a46c050c0ed840a8b696e,2021-08-15T00:00:00Z,2023-06-04T22:37:22Z,Matthew,Rogers,"9203 Jason Forge Maryview, MS 79947",2,WEBAPP,1,USA,1
28576200-0530-4370-ac21-5e1be93b9b6b,88ee42714f43e273f0deb5c29bd987e4a80f452c,2021-08-05T00:00:00Z,2023-06-08T19:56:16Z,William,Romero,"3501 Wright Ridge Suite 542 Johnsonside, MP 03305",6,WEBAPP,1,FR,1
43034f6b-8e2f-4895-877e-cffb44dfc26b,88ee42714f43e273f0deb5c29bd987e4a80f452c,2021-08-05T00:00:00Z,2023-06-08T19:56:16Z,William,Romero,"3501 Wright Ridge Suite 542 Johnsonside, MP 03305",1,WEBAPP,0,FR,0
25bf31a3-02e9-46e4-a591-6c1c476cc686,0b7a9e799fcb6b7809dec8ea158fa46627258c7d,2021-08-03T00:00:00Z,2023-06-04T13:42:34Z,Marissa,Wall,"68012 Tiffany Forges Mitchellberg, MN 48959",7,PHONE,1,USA,1
10f23980-4c69-437d-9816-e42bf3792acd,afb706c48c703a876568b2c1e1a6edd1a4d32fe8,2021-08-02T00:00:00Z,2023-06-01T03:34:03Z,Taylor,Harrison,Unit 0690 Box 4449 DPO AA 14319,3,WEBAPP,1,FR,1


In [0]:
%sql
ALTER TABLE spark_churn_users    SET TBLPROPERTIES (delta.autooptimize.optimizewrite = TRUE, delta.autooptimize.autocompact = TRUE );
ALTER TABLE spark_churn_orders   SET TBLPROPERTIES (delta.autooptimize.optimizewrite = TRUE, delta.autooptimize.autocompact = TRUE );
ALTER TABLE spark_churn_features SET TBLPROPERTIES (delta.autooptimize.optimizewrite = TRUE, delta.autooptimize.autocompact = TRUE );

## Our finale tables are now ready to be used to build SQL Dashboards and ML models for customer classification!
<img style="float: right" width="400" src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/retail/lakehouse-churn/lakehouse-retail-c360-dashboard-churn-prediction.png?raw=true"/>


Switch to Databricks SQL to see how this data can easily be requested using <a  dbdemos-dashboard-id="churn-universal" href='/sql/dashboardsv3/01f1b2d4419417ad8a9dcad32330b38a' target="_blank">Churn prediction DBSQL dashboard</a>, or an external BI tool. 

Creating a single flow was simple.  However, handling many data pipeline at scale can become a real challenge:
* Hard to build and maintain table dependencies 
* Difficult to monitor & enforce advance data quality
* Impossible to trace data lineage
* Difficult pipeline operations (observability, error recovery)


#### To solve these challenges, Databricks introduced **Spark Declarative Pipelines**
A simple way to build and manage data pipelines for fresh, high quality data!

# Next: secure and share data with Unity Catalog

Now that these tables are available in our Lakehouse, let's review how we can share them with the Data Scientists and Data Analysts teams.

Jump to the [Governance with Unity Catalog notebook]($../02-Data-governance/02-UC-data-governance-security-churn) or [Go back to the introduction]($../00-churn-introduction-lakehouse)